# Single-Turn Agent

In [4]:
from pydantic import BaseModel, Field

# from qdrant_client import QdrantClient
# from qdrant_client.models import Prefetch, Filter, FieldCondition, MatchText, FusionQuery

from langsmith import traceable, get_current_run_tree

from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode

from langchain_core.messages import AIMessage, ToolMessage

from jinja2 import Template
from typing import Literal, Dict, Any, Annotated, List, Optional
from IPython.display import Image, display
from operator import add
from openai import OpenAI
from pgvector.psycopg2 import register_vector

import openai

import psycopg2
import random
import ast
import inspect
import instructor
import json
import os
import sqlvalidator

In [7]:
postgres_username = os.getenv("POSTGRES_USERNAME")
postgres_pwd = os.getenv("POSTGRES_PASSWORD")

In [8]:
def get_postgres_conn():
    conn = psycopg2.connect(
        dbname="postgresdb",
        user=postgres_username,
        password=postgres_pwd,
        host="host.docker.internal",  # e.g., "localhost"
        port="5433"        # default PostgreSQL port
    )
    conn.autocommit = True
    with conn.cursor() as cur:
        cur.execute("CREATE EXTENSION IF NOT EXISTS vector;")
        conn.commit()
    register_vector(conn)  # Register pgvector type with psycopg2
    return conn

In [3]:
class SQLQuery(BaseModel):
    sql_command: str
    selected_columns: list


class RAGGenerationResponse(BaseModel):
    answer: str

# Helper

In [5]:
def is_valid_sql(query):
    parsed = sqlvalidator.parse(query)
    return parsed.is_valid()


def process_context(context):
    formatted_rows = "\n".join([", ".join(map(str, row)) for row in context])
    return formatted_rows

# Text2SQL Tool

In [ ]:
def get_embedding(text):
    result = openai.embeddings.create(
        input=[text],
        model="text-embedding-3-small"
    )
    embedding = result.data[0].embedding 
    return embedding

def generate_sql_query(question):
    prompt = f"""
    You are a PostgreSQL expert and you perform vector similarity search. 

    You only respond with PostgreSQL commands for the question asked by the user.

    You are given a database schema:
        Schema: public
        Table: us_attractions
        Columns:
        - id INTEGER PRIMARY KEY
        - name VARCHAR(250)
        - main_category VARCHAR(250)
        - rating REAL
        - reviews REAL
        - categories VARCHAR(250)
        - address VARCHAR(250)
        - city VARCHAR(250)
        - country VARCHAR(250)
        - state VARCHAR(250)
        - zipcode INTEGER
        - broader_category VARCHAR(250)
        - weighted_score REAL
        - weighted_average REAL
        - all_cities VARCHAR(250)
        - embedding VECTOR

    The us_attractions table only has information for USA only. The values under the country column are all 'USA'.

    Always include the 'id' column in the SQL command.

    Select a few relevant columns dynamically based on the question, such as id, name, rating, main_category, etc.

    Always include the following vector similarity comparison in your query to rank results by similarity:

    embedding::vector <=> %(embedding)s::vector AS distance

    where %(embedding)s is a placeholder. You are to leave the placeholder as instructed.

    Translate the following user question into PostgreSQL query statement:

    '{question}'

    Instructions:
    - Write PostgreSQL query using the "public" schema for all tables (e.g., public.us_attractions).
    - Order results by this distance (ascending, with closest matches first)
    - Limis results to 5 rows unless another limit is specified by the user
    - Does NOT include any WHERE clauses, filters, or other conditions because the vector similarity ranking fully determines relevance
    - Always returns the vector distance column named "distance"
    - If you cannot respond a PostgreSQL command respond with 'Sorry, no relevant data was found in the database for your query.'. Don't respond with anything else.
    """

    client = instructor.from_provider("groq/llama-3.3-70b-versatile")
    response, _ = client.chat.completions.create_with_completion(
        # model="gpt-4.1-mini",
        response_model=SQLQuery,
        messages=[{"role":"user", "content": prompt}],
        temperature=0
    )
    sql_command = response.sql_command
    return sql_command


def execute_sql_query(cursor, sql_query, params):
    rows = list()
    if is_valid_sql(sql_query):
        if params:
            cursor.execute(sql_query, params)
        else:
            cursor.execute(sql_query)
        rows = cursor.fetchall()
    return rows


def retrieve_context(query: str, top_k: int=5):
    pg_conn = get_postgres_conn()

    query_embedding = get_embedding(query)

    pg_params = {'embedding': query_embedding}
    sql_command = generate_sql_query(query)
    sql_query_results = execute_sql_query(pg_conn.cursor(), sql_command, pg_params)
    formatted_context = process_context(sql_query_results)
    return formatted_context
    

In [19]:
user_question = "Are there any art museums in Nashville?"

response = retrieve_context(user_question)

In [20]:
print(response)

1544, Frist Art Museum, Art museum, 4.6, Nashville, 0.35652748605444995
1554, National Museum of African American Music, Museum, 4.9, Nashville, 0.3724281600841328
1545, Tennessee State Museum, Museum, 4.7, Nashville, 0.3831187854338405
1539, The Parthenon, Art museum, 4.6, Nashville, 0.3868710994720459
1546, Madame Tussauds Nashville, Wax museum, 4.5, Nashville, 0.39156166365375034


# State and Pydantic Models

In [ ]:
class ToolCall(BaseModel):
    name: str
    arguments: dict

class RAGUsedContext(BaseModel):
    id: int
    description: str

class AgentResponse(BaseModel):
    answer: str
    tool_calls: List[ToolCall] = Field(default_factory=list)
    final_answer: bool = Field(default=False)
    retrieved_context_ids: List[RAGUsedContext]

class State(BaseModel):
    messages: Annotated[List[Any], add] = []
    answer: str = ""
    iteration: int = Field(default=0)
    final_answer: bool = Field(default=False)
    available_tools: List[Dict[str, Any]] = []
    tool_calls: Optional[List[ToolCall]] = Field(default_factory=list)
    retrieved_context_ids: Annotated[List[RAGUsedContext], add] = []

# Agent Node

In [ ]:
def generate_sql_query(question, limit=5):
    schema_info = """
    Schema: public.us_attractions
    Columns:
    - id INTEGER PRIMARY KEY
    - name VARCHAR(250)
    - main_category VARCHAR(250)
    - rating REAL
    - reviews REAL
    - categories VARCHAR(250)
    - address VARCHAR(250)
    - city VARCHAR(250)
    - country VARCHAR(250)
    - state VARCHAR(250)
    - zipcode INTEGER
    - broader_category VARCHAR(250)
    - weighted_score REAL
    - weighted_average REAL
    - all_cities VARCHAR(250)
    - embedding VECTOR
    """

    few_shot_examples = """
    User question: "Find top 5 attractions with high ratings in New York."
    SQL query:
    SELECT id, name, rating, city, embedding <=> %(embedding)s::vector AS distance
    FROM public.us_attractions
    ORDER BY distance ASC
    LIMIT 5;

    User question: "Show attractions with the best reviews."
    SQL query:
    SELECT id, name, reviews, rating, embedding <=> %(embedding)s::vector AS distance
    FROM public.us_attractions
    ORDER BY distance ASC
    LIMIT 5;
    """

    prompt = f"""
    You are a PostgreSQL expert specializing in vector similarity search.

    Given the table schema below:
    {schema_info}

    Translate the following natural language question into a PostgreSQL SQL query that:
    - Selects relevant columns dynamically including id, name, and other columns relevant to the question.
    - Computes vector similarity using: embedding <=> %(embedding)s::vector AS distance
    - Orders results by ascending distance.
    - Limits results to {limit} rows.
    - Always includes the 'distance' column in the result.
    - Do NOT include any non-vector filters (no WHERE clauses).

    Examples:
    {few_shot_examples}

    User question: "{question}"

    SQL query:
    """

    client = instructor.from_provider("groq/llama-3.3-70b-versatile")
    response, _ = client.chat.completions.create_with_completion(
        response_model=SQLQuery,
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    sql_command = response.sql_command
    return sql_command


In [ ]:
def agent(state: MessagesState) -> Command[Literal["vector_sql_search", "web_search", "direct_answer", "END"]]:
    # Retrieve relevant data from state, e.g., the latest user question
    user_question = state["messages"][-1] if state.get("messages") else ""

    # Craft a prompt instructing the model to pick the best tool for the question
    prompt = f"""
You are an intelligent assistant deciding how to handle the user's question.

Tools you can use:
- vector_sql_search: Perform a vector similarity search in the US attractions database.
- web_search: Search the web for recent information.
- direct_answer: Generate an answer directly from your own knowledge.

User question: "{user_question}"

Respond only in JSON format with the fields:
{{"next_agent": "one of vector_sql_search, web_search, direct_answer, or END", "reason": "brief rationale"}}
"""

    # Call the LLM with the prompt
    response_text = model.invoke(messages=[{"role": "system", "content": prompt}])

    # Parse the model JSON response
    try:
        response_json = json.loads(response_text)
        next_agent = response_json.get("next_agent", "END")
    except Exception:
        next_agent = "END"

    # Return a Command to route to next agent node
    return Command(goto=next_agent, update={"reason": response_text})